In [1]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

PATH = "checkpoints/awq_models/law_llm_awq_w4a16"

# vLLM auto-detects compressed-tensors W4A16 from the checkpoint config
llm = LLM(model=PATH, dtype="auto", 
            gpu_memory_utilization=0.20, 
            max_model_len=4096)

tokenizer = AutoTokenizer.from_pretrained(PATH)

/home/lisa/Arupreza/PRISM-RAG/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/lisa/Arupreza/PRISM-RAG/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 06-03 16:58:29 [__init__.py:244] Automatically detected platform cuda.
INFO 06-03 16:58:35 [config.py:841] This model supports multiple tasks: {'embed', 'generate', 'reward', 'classify'}. Defaulting to 'generate'.
INFO 06-03 16:58:35 [config.py:3368] Downcasting torch.float32 to torch.bfloat16.
INFO 06-03 16:58:35 [config.py:1472] Using max model len 4096


2026-06-03 16:58:35,251	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 06-03 16:58:35 [config.py:2285] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 06-03 16:58:35 [core.py:526] Waiting for init message from front-end.
INFO 06-03 16:58:35 [core.py:69] Initializing a V1 LLM engine (v0.9.2) with config: model='checkpoints/awq_models/law_llm_awq_w4a16', speculative_config=None, tokenizer='checkpoints/awq_models/law_llm_awq_w4a16', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=compressed-tensors, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_backend=''), observability_config=ObservabilityConfig(show_

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.62it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.62it/s]



INFO 06-03 16:58:37 [default_loader.py:272] Loading weights took 0.69 seconds
INFO 06-03 16:58:37 [gpu_model_runner.py:1801] Model loading took 3.8562 GiB and 0.890591 seconds
INFO 06-03 16:58:44 [backends.py:508] Using cache directory: /home/lisa/.cache/vllm/torch_compile_cache/55498ffbc0/rank_0_0/backbone for vLLM's torch.compile
INFO 06-03 16:58:44 [backends.py:519] Dynamo bytecode transform time: 6.78 s
INFO 06-03 16:58:48 [backends.py:155] Directly load the compiled graph(s) for shape None from the cache, took 3.738 s
INFO 06-03 16:58:49 [monitor.py:34] torch.compile takes 6.78 s in total
INFO 06-03 16:58:51 [gpu_worker.py:232] Available KV cache memory: 4.66 GiB
INFO 06-03 16:58:51 [kv_cache_utils.py:716] GPU KV cache size: 38,128 tokens
INFO 06-03 16:58:51 [kv_cache_utils.py:720] Maximum concurrency for 4,096 tokens per request: 9.27x


Capturing CUDA graph shapes: 100%|██████████| 67/67 [00:17<00:00,  3.80it/s]


INFO 06-03 16:59:09 [gpu_model_runner.py:2326] Graph capturing finished in 18 secs, took 0.49 GiB
INFO 06-03 16:59:09 [core.py:172] init engine (profile, create kv cache, warmup model) took 32.00 seconds


In [2]:
SYS = ("You are a legal assistant. Answer the question using ONLY the context. "
        "Cite the relevant clause. If the answer is not in the context, say so.")

sp = SamplingParams(temperature=0.0, max_tokens=256)   # greedy, deterministic

def answer(context, question):
    user = f"{SYS}\n\nContext:\n{context}\n\nQuestion: {question}"
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": user}], tokenize=False, add_generation_prompt=True
    )
    out = llm.generate([prompt], sp)
    return out[0].outputs[0].text.strip()

In [3]:
ctx = ("Section 7.2. Termination for Convenience. Either party may terminate this "
       "Agreement without cause upon ninety (90) days prior written notice to the "
       "other party. Section 7.3. Termination for Cause. A party may terminate "
       "immediately upon material breach that remains uncured for thirty (30) days.")

print(answer(ctx, "How much notice is required to terminate this agreement for convenience?"))

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.86it/s, est. speed input: 253.65 toks/s, output: 76.46 toks/s]

To terminate this agreement for convenience, a party must provide 90 days prior written notice to the other party. This information can be found in Section 7.2 of the provided context.


In [4]:
print(answer(ctx, "What is the governing law of this agreement?"))

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  8.03it/s, est. speed input: 1066.42 toks/s, output: 129.25 toks/s]

The governing law of this agreement is not explicitly mentioned in the provided context.


In [ ]:
ctx_irrelevant = ("Section 3.1. The Supplier shall deliver the Goods to the Buyer's "
                "warehouse at 14 Industrial Road by the Delivery Date.")
print(answer(ctx_irrelevant, "What are the elements of promissory estoppel?"))

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  7.02it/s, est. speed input: 677.63 toks/s, output: 127.04 toks/s]

The elements of promissorial estoppel are not mentioned in the provided context.
